In [1]:
import pandas as pd
import numpy as np

In [3]:
# 1. Load the Data
df = pd.read_csv('india_housing_prices.csv')

In [4]:
df.head()

,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  object 
 2   City                            250000 non-null  object 
 3   Locality                        250000 non-null  object 
 4   Property_Type                   250000 non-null  object 
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  object 
 11  Floor_No                        250000 non-null  int64  
 12  Total_Floors    

In [6]:
df.columns

Index(['ID', 'State', 'City', 'Locality', 'Property_Type', 'BHK',
       'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Year_Built',
       'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property',
       'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility',
       'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type',
       'Availability_Status'],
      dtype='object')

In [8]:
df.describe()

,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000
mean,125000.500000,2.999396,2749.813216,254.586854,0.130597,2006.520012,14.966800,15.503004,18.479988,5.499860,5.498016
std,72168.927986,1.415521,1300.606954,141.349921,0.130747,9.808575,8.948047,8.671618,9.808575,2.878639,2.871860
min,1.000000,1.000000,500.000000,10.000000,0.000000,1990.000000,0.000000,1.000000,2.000000,1.000000,1.000000
25%,62500.750000,2.000000,1623.000000,132.550000,0.050000,1998.000000,7.000000,8.000000,10.000000,3.000000,3.000000
50%,125000.500000,3.000000,2747.000000,253.870000,0.090000,2007.000000,15.000000,15.000000,18.000000,5.000000,5.000000
75%,187500.250000,4.000000,3874.000000,376.880000,0.160000,2015.000000,23.000000,23.000000,27.000000,8.000000,8.000000
max,250000.000000,5.000000,5000.000000,500.000000,0.990000,2023.000000,30.000000,30.000000,35.000000,10.000000,10.000000


In [9]:
# 2. Data Cleaning
# Remove rows where Floor Number is impossible (e.g. Floor 25 in a 5-story building)
df_clean = df[df['Floor_No'] <= df['Total_Floors']].copy()
# Remove rows with invalid prices
df_clean = df_clean[df_clean['Price_per_SqFt'] > 0]

In [11]:
# 3. Feature Engineering: Future Price (Regression Target)
# Formula: Future = Current * (1 + r)^t
growth_rate = 0.08 # 8% annual growth
years = 5
df_clean['Future_Price_5Y'] = df_clean['Price_in_Lakhs'] * ((1 + growth_rate) ** years)
df_clean['Future_Price_5Y'] = df_clean['Future_Price_5Y'].round(2)

In [12]:
# 4. Feature Engineering: Good Investment (Classification Target)
# Calculate the Median Price for each Locality
locality_stats = df_clean.groupby('Locality')['Price_in_Lakhs'].median().reset_index()
locality_stats.rename(columns={'Price_in_Lakhs': 'Median_Locality_Price'}, inplace=True)

In [13]:
# Merge this median info back into the main dataset
df_clean = pd.merge(df_clean, locality_stats, on='Locality', how='left')

In [14]:
# Logic: If price is less than or equal to the locality median, it's a "Good Investment"
df_clean['Good_Investment'] = np.where(df_clean['Price_in_Lakhs'] <= df_clean['Median_Locality_Price'], 1, 0)

In [16]:
# 5. Save the processed file
df_clean.to_csv('processed_housing_data.csv', index=False)
print("Success! Data processed and saved as 'processed_housing_data.csv'.")

Success! Data processed and saved as 'processed_housing_data.csv'.


In [17]:
df = pd.read_csv('processed_housing_data.csv')

In [18]:
# --- TASK 1: Handle Duplicates ---
initial_count = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_count - len(df)} duplicate rows.")

Removed 0 duplicate rows.


In [19]:
# --- TASK 2: Create 'School Density Score' ---
# The logic: We will categorize the 'Nearby_Schools' count into a readable score/category
# High Density = 8+ schools, Medium = 4-7 schools, Low = 0-3 schools
def categorize_schools(count):
    if count >= 8:
        return 'High Density'
    elif count >= 4:
        return 'Medium Density'
    else:
        return 'Low Density'

df['School_Density_Score'] = df['Nearby_Schools'].apply(categorize_schools)

In [20]:
# --- TASK 3: Ensure 'Price per SqFt' is clean ---
# (It was already in the data, but let's ensure it matches our clean data)
df['Price_per_SqFt'] = df['Price_per_SqFt'].round(2)

In [21]:
# Save the final version
df.to_csv('processed_housing_data_final.csv', index=False)
print("Checklist Complete! Final data saved as 'processed_housing_data_final.csv'")

Checklist Complete! Final data saved as 'processed_housing_data_final.csv'


In [22]:
print(df[['Locality', 'Nearby_Schools', 'School_Density_Score']].head())

       Locality  Nearby_Schools School_Density_Score
0  Locality_167               9         High Density
1  Locality_393               5       Medium Density
2  Locality_122               3          Low Density
3   Locality_75               1          Low Density
4  Locality_462               6       Medium Density


In [2]:
# Load the processed data 
df = pd.read_csv('processed_housing_data_final.csv')

In [3]:
# Remove zero or negative values
df = df[df['Size_in_SqFt'] > 0]
df = df[df['Price_in_Lakhs'] > 0]
df = df[df['Price_per_SqFt'] > 0]

In [4]:
# Remove extreme outliers using IQR
Q1 = df['Price_in_Lakhs'].quantile(0.25)
Q3 = df['Price_in_Lakhs'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['Price_in_Lakhs'] >= Q1 - 1.5 * IQR) & (df['Price_in_Lakhs'] <= Q3 + 1.5 * IQR)]


In [11]:
# Save the final version
df.to_csv('processed_housing_data_final2.csv', index=False)
print("Checklist Complete! Final data saved as 'processed_housing_data_final2.csv'")

Checklist Complete! Final data saved as 'processed_housing_data_final2.csv'


In [6]:
df = pd.read_csv('processed_housing_data_final2.csv')

In [7]:
# Remove zero or missing values
df = df[(df['Size_in_SqFt'] > 0) & (df['Price_in_Lakhs'] > 0)]

# Use IQR method for both price and size
for col in ['Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

In [8]:
df[['Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt']].corr()

,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt
Size_in_SqFt,1.000000,0.153268,-0.544353
Price_in_Lakhs,0.153268,1.000000,0.625397
Price_per_SqFt,-0.544353,0.625397,1.000000


In [9]:
df.groupby('Property_Type')[['Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt']].corr()

Size_in_SqFt  Price_in_Lakhs  Price_per_SqFt
Property_Type                                                                 
Apartment         Size_in_SqFt        1.000000        0.151122       -0.545247
                  Price_in_Lakhs      0.151122        1.000000        0.625503
                  Price_per_SqFt     -0.545247        0.625503        1.000000
Independent House Size_in_SqFt        1.000000        0.162019       -0.539947
                  Price_in_Lakhs      0.162019        1.000000        0.623795
                  Price_per_SqFt     -0.539947        0.623795        1.000000
Villa             Size_in_SqFt        1.000000        0.146714       -0.547827
                  Price_in_Lakhs      0.146714        1.000000        0.626884
                  Price_per_SqFt     -0.547827        0.626884        1.000000

In [10]:
df.groupby('City')[['Size_in_SqFt', 'Price_in_Lakhs']].corr()

Size_in_SqFt  Price_in_Lakhs
City                                                       
Ahmedabad      Size_in_SqFt        1.000000        0.170976
               Price_in_Lakhs      0.170976        1.000000
Amritsar       Size_in_SqFt        1.000000        0.141045
               Price_in_Lakhs      0.141045        1.000000
Bangalore      Size_in_SqFt        1.000000        0.160428
...                                     ...             ...
Vijayawada     Price_in_Lakhs      0.162929        1.000000
Vishakhapatnam Size_in_SqFt        1.000000        0.148248
               Price_in_Lakhs      0.148248        1.000000
Warangal       Size_in_SqFt        1.000000        0.160790
               Price_in_Lakhs      0.160790        1.000000

[84 rows x 2 columns]